# Rung 3 — Do the data distinguish the parameters?

**Question.** Rung 2 found a valley. How flat is it, in which direction, and what does
that mean for what we can claim?

Three views of one idea, in increasing cost and honesty: the **sensitivity spectrum**
(how the predictions stretch with the parameters), the **observed Fisher information**
(the curvature of the likelihood, which brings in the noise), and the **profile
likelihood** (re-optimising everything else, which is the only one that makes no
quadratic assumption).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset, default_sir_model
from inferladder.diagnostics import profile_likelihood, sensitivity_spectrum
from inferladder.inference import fit, multistart, negative_log_likelihood, random_starts

dataset = default_sir_dataset()
model = default_sir_model()
objective = negative_log_likelihood(model, dataset.y, dataset.t)
mle = fit(objective, dataset.theta_true * 1.2, model.param_names)
mle.as_dict

## The sensitivity spectrum

Singular values of the Jacobian of the predictions with respect to the parameters, in
log-log units so the entries are elasticities and comparable across parameters.

In [ ]:
spectrum = sensitivity_spectrum(
    lambda theta: model.expected(theta, dataset.t), mle.theta, model.param_names
)
print("singular values:", np.array2string(spectrum.singular_values, precision=4))
print(f"condition number: {spectrum.condition_number:.3g}")
print("\nstiffest direction :", {k: round(v, 3) for k, v in spectrum.stiffest().items()})
print("sloppiest direction:", {k: round(v, 3) for k, v in spectrum.sloppiest().items()})

The last singular value is exactly zero, and its direction is pure `k`. That is not a
numerical accident and not a defect: `k` is the dispersion of the negative binomial, so it
changes the *spread* of the data and not the *mean*. A spectrum built from the predicted
mean is structurally blind to it.

It is a useful reminder of what this diagnostic does and does not see. A noise parameter
needs the curvature of the likelihood, not the sensitivity of the mean. So restrict the
spectrum to the parameters that move the mean:

In [ ]:
mean_params = model.param_names[:3]  # beta, gamma, rho
sub = sensitivity_spectrum(
    lambda theta: model.expected(np.append(theta, mle.theta[3]), dataset.t),
    mle.theta[:3],
    mean_params,
)
print("singular values:", np.array2string(sub.singular_values, precision=4))
print(f"condition number: {sub.condition_number:.1f}")
print("\nstiffest :", {k: round(v, 3) for k, v in sub.stiffest().items()})
print("sloppiest:", {k: round(v, 3) for k, v in sub.sloppiest().items()})

## Profile likelihood

The expensive, honest diagnostic: fix one parameter, re-optimise all the others, and
record the best fit achievable. Unlike the slice in rung 2 this lets the other parameters
compensate, which is the difference between a parameter that is genuinely pinned down and
one that only looks pinned down.

In [ ]:
threshold = 1.92  # half the 95% chi-squared quantile with 1 degree of freedom

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
profiles = {}
for ax, name in zip(axes, ("beta", "gamma"), strict=True):
    index = model.param_names.index(name)
    centre = mle.theta[index]
    grid = np.linspace(centre * 0.82, centre * 1.22, 13)
    prof = profile_likelihood(objective, mle.theta, index, grid, model.param_names)
    profiles[name] = prof

    ax.plot(prof.values, prof.delta, "-o", ms=3, color="#33628f")
    ax.axhline(threshold, ls="--", color="#c1502e", label="95% threshold")
    ax.axvline(dataset.truth[name], ls=":", color="#3c3c3c", label="truth")
    ax.set(xlabel=name, ylabel="rise above best fit", ylim=(0, 6))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Profile likelihoods: both parameters are bounded on the full curve")
fig.tight_layout()

Both profiles rise through the threshold on either side, so on the *complete* epidemic
both rates are individually identified. The valley of rung 2 is long but not infinite.

## The ridge is a property of the window, not of the model

Now repeat the multistart of rung 2 using only the first 25 days — the exponential phase,
which is what rung 0 fitted and what an analyst has in hand *during* an outbreak.

In [ ]:
early = slice(0, 25)
early_objective = negative_log_likelihood(model, dataset.y[early], dataset.t[early])
_, early_runs = multistart(
    early_objective, random_starts(dataset.theta_true, 12, spread=0.6, seed=2), model.param_names
)
early_values = np.array([r.objective_value for r in early_runs])
converged = np.array([r.theta for r in early_runs])[early_values < early_values.min() + 0.5]

print(f"{'quantity':12} {'min':>12} {'max':>14}")
for label, column in (
    ("beta", converged[:, 0]),
    ("gamma", converged[:, 1]),
    ("beta - gamma", converged[:, 0] - converged[:, 1]),
    ("R0", converged[:, 0] / converged[:, 1]),
):
    print(f"{label:12} {column.min():12.4f} {column.max():14.4f}")

This is the ridge, in its sharpest form. Twelve searches agree on `beta - gamma` to
four decimals and disagree about `R0` by **orders of magnitude**. `gamma` slides to zero
and `beta` follows it; the difference is fixed, the ratio is unconstrained.

It is worth stating the consequence plainly, because it inverts a common expectation:

| what is observed | what is well determined | what is not |
|---|---|---|
| early exponential phase only | `beta - gamma` (the growth rate) | `R0`, and each rate separately |
| the complete epidemic | each rate, and `R0` | — |

`R0` is not intrinsically the rigid quantity. It becomes identified only once the curve
has bent — that is, once susceptible depletion and the final size have been observed. An
`R0` quoted from the exponential phase alone is carrying an assumption about `gamma` that
the data did not supply.

## When to stop here

**This rung is enough to know what you are allowed to claim.** It separates "the model
fits" from "the data determine this parameter", which is the distinction that makes a
fitted parameter safe to quote.

**It still gives no interval.** Everything here is about shape. Turning that shape into a
number with an error bar is rung 4.